In [ ]:
import os
import zipfile
from glob import glob
from tqdm import tqdm

# ---------------- 사용자 설정 ----------------
input_zip_dir = r"<DATA_ROOT>\07_disaster\wildfire\hawaii\sentinel2"
output_unzip_dir = r"<DATA_ROOT>\07_disaster\wildfire\hawaii\sentinel2\unzipped"
os.makedirs(output_unzip_dir, exist_ok=True)
# --------------------------------------------

def unzip_all_sentinel2_zips(input_dir, output_dir):
    zip_files = glob(os.path.join(input_dir, "*.zip"))
    
    for zip_file in tqdm(zip_files, desc="Unzipping all .zip files"):
        zip_name = os.path.splitext(os.path.basename(zip_file))[0]
        target_folder = os.path.join(output_dir, zip_name)

        if os.path.exists(target_folder):
            print(f"[스킵] 이미 압축 해제됨: {zip_name}")
            continue

        os.makedirs(target_folder, exist_ok=True)

        with zipfile.ZipFile(zip_file, 'r') as zip_ref:
            zip_ref.extractall(target_folder)

        print(f"[완료] 압축 해제됨: {zip_name}")

# 실행
unzip_all_sentinel2_zips(input_zip_dir, output_unzip_dir)


In [ ]:
# Sentinel-2 광학지수 산출

import os
import rasterio
import numpy as np 
from rasterio.warp import reproject, Resampling
from glob import glob
from tqdm import tqdm

# ---------------- 사용자 설정 ----------------
unzipped_root = r"<DATA_ROOT>\07_disaster\wildfire\Greece\Evia\sentinel2"
output_dir = r"<DATA_ROOT>\07_disaster\wildfire\Greece\Evia\process"
target_crs = 'EPSG:32634'
# --------------------------------------------

def get_band_path(safe_folder, band_code, resolution="10m"):
    # 중첩된 SAFE 폴더 내부 탐색
    sub_safe = glob(os.path.join(safe_folder, "*.SAFE"))[0]
    pattern = os.path.join(sub_safe, "GRANULE", "*", "IMG_DATA", f"R{resolution}", f"*_{band_code}_{resolution}.jp2")
    matches = glob(pattern)
    if not matches:
        print(f"[경고] {band_code} 밴드를 찾을 수 없습니다: {pattern}")
    return matches[0] if matches else None

def resample_to_10m(src_path, ref_path):
    with rasterio.open(ref_path) as ref:
        ref_transform = ref.transform
        ref_width = ref.width
        ref_height = ref.height
        ref_crs = ref.crs
        ref_profile = ref.profile

    with rasterio.open(src_path) as src:
        data = src.read(1)
        dst = np.empty((ref_height, ref_width), dtype=np.float32)
        reproject(
            source=data,
            destination=dst,
            src_transform=src.transform,
            src_crs=src.crs,
            dst_transform=ref_transform,
            dst_crs=ref_crs,
            resampling=Resampling.bilinear
        )
    return dst, ref_profile

def calculate_index(numerator, denominator):
    with np.errstate(divide='ignore', invalid='ignore'):
        result = (numerator - denominator) / (numerator + denominator)
        result[np.isinf(result)] = np.nan
    return result

def save_geotiff(path, data, profile):
    profile.update({
        'driver': 'GTiff',
        'dtype': 'float32',
        'count': 1,
        'crs': target_crs
    })
    with rasterio.open(path, 'w', **profile) as dst:
        dst.write(data, 1)

def save_rgb(path, bands, profile):
    profile.update({
        'driver': 'GTiff',
        'dtype': 'uint16',
        'count': 3,
        'crs': target_crs
    })
    with rasterio.open(path, 'w', **profile) as dst:
        for i in range(3):
            dst.write(bands[i], i+1)

def extract_scene_name(safe_folder):
    full_name = os.path.basename(safe_folder)
    parts = full_name.split('_')
    date_part = parts[2][:8] if len(parts) > 2 else "YYYYMMDD"
    tile_part = [p for p in parts if p.startswith('T') and len(p) == 6]
    tile_id = tile_part[0] if tile_part else "TILE"
    return f"{date_part}_{tile_id}"

# ---------------- 본 처리 루프 ----------------
safe_folders = glob(os.path.join(unzipped_root, "*.SAFE"))

for safe_folder in tqdm(safe_folders, desc="Sentinel-2 처리 중"):
    scene_name = extract_scene_name(safe_folder)

    # 밴드 경로 가져오기
    B02 = get_band_path(safe_folder, "B02", "10m")
    B03 = get_band_path(safe_folder, "B03", "10m")
    B04 = get_band_path(safe_folder, "B04", "10m")
    B06 = get_band_path(safe_folder, "B06", "20m")
    B07 = get_band_path(safe_folder, "B07", "20m")
    B08 = get_band_path(safe_folder, "B08", "10m")
    B8A = get_band_path(safe_folder, "B8A", "20m")
    B11 = get_band_path(safe_folder, "B11", "20m")
    B12 = get_band_path(safe_folder, "B12", "20m")

    if None in [B02, B03, B04, B06, B07, B08, B8A, B11, B12]:
        print(f"[스킵] {scene_name} 처리 불가 - 필수 밴드 누락")
        continue

    # 10m 밴드 읽기
    with rasterio.open(B04) as ref:
        profile = ref.profile
        red = ref.read(1)
    with rasterio.open(B03) as f: green = f.read(1)
    with rasterio.open(B02) as f: blue = f.read(1)
    with rasterio.open(B08) as f: nir = f.read(1)

    # 20m 밴드 리샘플링
    swir1, _ = resample_to_10m(B11, B04)
    swir2, _ = resample_to_10m(B12, B04)
    b06, _ = resample_to_10m(B06, B04)
    b07, _ = resample_to_10m(B07, B04)
    b8a, _ = resample_to_10m(B8A, B04)

    # TrueColor
    save_rgb(os.path.join(output_dir, f"{scene_name}_TrueColor.tif"), [red, green, blue], profile)

    # NDVI
    ndvi = calculate_index(nir.astype(np.float32), red.astype(np.float32))
    save_geotiff(os.path.join(output_dir, f"{scene_name}_NDVI.tif"), ndvi, profile)

    # NBR
    nbr = calculate_index(nir.astype(np.float32), swir2)
    save_geotiff(os.path.join(output_dir, f"{scene_name}_NBR.tif"), nbr, profile)

    # NDBI
    ndbi = calculate_index(swir1, nir.astype(np.float32))
    save_geotiff(os.path.join(output_dir, f"{scene_name}_NDBI.tif"), ndbi, profile)

    # FalseColor
    save_rgb(os.path.join(output_dir, f"{scene_name}_FalseColor.tif"), [swir2.astype(np.uint16), swir1.astype(np.uint16), red], profile)

    # BAIS2
    with np.errstate(divide='ignore', invalid='ignore'):
        # 반사율로 변환
        red_f   = red.astype(np.float32) / 10000.0
        b06_f   = b06 / 10000.0
        b07_f   = b07 / 10000.0
        b8a_f  = b8a / 10000.0
        b12_f   = swir2 / 10000.0  # swir2 == B12

        numerator = 1 - np.sqrt((b06_f * b07_f * b8a_f) / red_f)
        denominator = np.sqrt(b12_f + b8a_f)
        bais2 = numerator * (((b12_f - b8a_f) / denominator) + 1)

        bais2[np.isinf(bais2)] = np.nan

    save_geotiff(
        os.path.join(output_dir, f"{scene_name}_BAIS2.tif"),
        bais2,
        profile
    )


print("✅ 모든 GeoTIFF 저장 완료!")


In [ ]:
# Sentinel-2 2개 영상 dNBR 산출

import rasterio
import numpy as np
import os

# ---------------- 사용자 설정 ----------------
nbr_before_path = r"<DATA_ROOT>\07_disaster\wildfire\Greece\Evia\process\20250725_T34SGH_NBR.tif"
nbr_after_path  = r"<DATA_ROOT>\07_disaster\wildfire\Greece\Evia\process\20250728_T34SGH_NBR.tif"
output_path     = r"<DATA_ROOT>\07_disaster\wildfire\Greece\Evia\process\dNBR_20250725_20250728.tif"
# --------------------------------------------

# 두 NBR TIF 불러오기
with rasterio.open(nbr_before_path) as src_before:
    nbr_before = src_before.read(1).astype(np.float32)
    profile = src_before.profile

with rasterio.open(nbr_after_path) as src_after:
    nbr_after = src_after.read(1).astype(np.float32)

# dNBR 계산
dnbr = nbr_before - nbr_after

# 저장
profile.update({
    'driver': 'GTiff',
    'dtype': 'float32',
    'count': 1
})
with rasterio.open(output_path, 'w', **profile) as dst:
    dst.write(dnbr, 1)

print(f"✅ dNBR 저장 완료: {output_path}")


In [ ]:
# Sentinel-2 2개 영상 dNDBI 산출

import rasterio
import numpy as np
import os

# ---------------- 사용자 설정 ----------------
NDBI_before_path = r"<DATA_ROOT>\07_disaster\wildfire\hawaii\process\20230729_T04QGJ_NDBI.tif"
NDBI_after_path  = r"<DATA_ROOT>\07_disaster\wildfire\hawaii\process\20230818_T04QGJ_NDBI.tif"
output_path     = r"<DATA_ROOT>\07_disaster\wildfire\hawaii\process\dNDBI_20230729_20230818.tif"
# --------------------------------------------

# 두 NDBI TIF 불러오기
with rasterio.open(NDBI_before_path) as src_before:
    NDBI_before = src_before.read(1).astype(np.float32)
    profile = src_before.profile

with rasterio.open(NDBI_after_path) as src_after:
    NDBI_after = src_after.read(1).astype(np.float32)

# dNDBI 계산
dNDBI = NDBI_before - NDBI_after

# 저장
profile.update({
    'driver': 'GTiff',
    'dtype': 'float32',
    'count': 1
})
with rasterio.open(output_path, 'w', **profile) as dst:
    dst.write(dNDBI, 1)

print(f"✅ dNDBI 저장 완료: {output_path}")


In [ ]:
# Planet Scope NDVI 산출

import rasterio
import numpy as np

# ---------------- 사용자 설정 ----------------
input_path = r"<DATA_ROOT>\07_disaster\wildfire\hawaii\PlanetScope\hawaii2_psscene_analytic_8b_sr_udm2\PSScene\20230811_204519_78_247c_3B_AnalyticMS_SR_8b_clip.tif"
output_path = r"<DATA_ROOT>\07_disaster\wildfire\hawaii\process\planet_20230811_NDVI.tif"
# ---------------------------------------------

# 밴드 6 = Red, 밴드 8 = NIR
with rasterio.open(input_path) as src:
    red = src.read(6).astype(np.float32)
    nir = src.read(8).astype(np.float32)
    profile = src.profile

# NDVI 계산
ndvi = (nir - red) / (nir + red + 1e-6)

# GeoTIFF로 저장
profile.update({
    'driver': 'GTiff',
    'dtype': 'float32',
    'count': 1
})
with rasterio.open(output_path, 'w', **profile) as dst:
    dst.write(ndvi, 1)

print(f"✅ NDVI 저장 완료: {output_path}")


In [ ]:
# median NBR
# dNBR (median NBR - 각 날짜)
# 산불 또는 피해지역 추출 (임계값 설정)


import os
import glob
import numpy as np
import rasterio
from tqdm import tqdm

# ---------------- 사용자 입력 설정 ----------------
nbr_dir = r"<DATA_ROOT>\07_disaster\wildfire\hawaii\process"

USE_NBR_THRESHOLD = True        # True로 설정 시 NBR < NBR_THRESHOLD 조건 사용
NBR_THRESHOLD = -0.2

USE_DNBR_THRESHOLD = True       # True로 설정 시 dNBR > DNBR_THRESHOLD 조건 사용
DNBR_THRESHOLD = 0.2
# --------------------------------------------------

output_dir = nbr_dir
nbr_files = sorted(glob.glob(os.path.join(nbr_dir, "*_NBR.tif")))
nbr_files = [f for f in nbr_files if not os.path.basename(f).startswith("median_")]
assert len(nbr_files) > 0, "NBR tif 파일이 없습니다."

# ---------------- median NBR 계산 ----------------
median_path = os.path.join(output_dir, "median_NBR.tif")
if os.path.exists(median_path):
    with rasterio.open(median_path) as src:
        median_nbr = src.read(1).astype(np.float32)
        profile = src.profile
    print("✅ median_NBR.tif 파일이 이미 존재하여 불러옴")
else:
    stack = []
    profile = None
    for path in nbr_files:
        with rasterio.open(path) as src:
            arr = src.read(1).astype(np.float32)
            arr[arr <= -9999] = np.nan
            stack.append(arr)
            if profile is None:
                profile = src.profile
    stack = np.stack(stack, axis=0)
    median_nbr = np.nanmedian(stack, axis=0)

    profile.update(dtype='float32', count=1)
    with rasterio.open(median_path, 'w', **profile) as dst:
        dst.write(median_nbr, 1)
    print("✅ median_NBR.tif 계산 및 저장 완료")

# ---------------- dNBR 계산 및 마스크 ----------------
for path in tqdm(nbr_files, desc="dNBR 계산 중"):
    fname = os.path.basename(path)
    date_str = fname.split("_")[0]
    dnbr_path = os.path.join(output_dir, f"{date_str}_dNBR.tif")

    if os.path.exists(dnbr_path):
        with rasterio.open(dnbr_path) as src:
            dnbr = src.read(1).astype(np.float32)
            profile = src.profile
        print(f"⏭️ {date_str}_dNBR.tif 이미 존재 - 불러오기")
    else:
        with rasterio.open(path) as src:
            nbr = src.read(1).astype(np.float32)
            nbr[nbr <= -9999] = np.nan          # nodata 마스킹
        dnbr = median_nbr - nbr
        with rasterio.open(dnbr_path, 'w', **profile) as dst:
            dst.write(dnbr, 1)
        print(f"✅ {date_str}_dNBR.tif 저장 완료")

    ###### 여기부터 주석 ######
    mask = np.ones_like(dnbr, dtype=np.uint8)
    condition_str = []

    if USE_NBR_THRESHOLD:
        with rasterio.open(path) as src:
            nbr = src.read(1).astype(np.float32)
            nbr[nbr <= -9999] = np.nan
        mask &= (nbr < NBR_THRESHOLD)
        condition_str.append(f"NBR{NBR_THRESHOLD}")

    if USE_DNBR_THRESHOLD:
        mask &= (dnbr > DNBR_THRESHOLD)
        condition_str.append(f"DNBR{DNBR_THRESHOLD}")

    if condition_str:
        binary_filename = f"{date_str}_binary_" + "_".join(condition_str) + ".tif"
        binary_path = os.path.join(output_dir, binary_filename)

        if os.path.exists(binary_path):
            print(f"⏭️ {binary_filename} 이미 존재 - 건너뜀")
        else:
            profile_mask = profile.copy()
            profile_mask.update(dtype='uint8')
            with rasterio.open(binary_path, 'w', **profile_mask) as dst:
                dst.write(mask, 1)
            print(f"✅ {binary_filename} 저장 완료")

print("🎯 모든 dNBR 및 바이너리 마스크 저장 완료")


In [ ]:
# 기준날짜 - 대상날짜 dNBR 산출

import os
import glob
import re
import numpy as np
import rasterio
from rasterio.enums import Resampling
from rasterio.transform import Affine

# ---------------- 사용자 설정 ----------------
nbr_dir = r"<DATA_ROOT>\07_disaster\wildfire\hawaii\process"
ref_path = os.path.join(nbr_dir, "20230803_T04QGJ_NBR.tif")  # 기준 NBR
# -------------------------------------------

# 날짜 추출용 정규식: 앞 8자리(YYYYMMDD)
date_re = re.compile(r"(\d{8})")

# 기준 영상 열기
with rasterio.open(ref_path) as ref_src:
    ref_nbr = ref_src.read(1, masked=True)
    ref_meta = ref_src.meta.copy()
    ref_date = date_re.search(os.path.basename(ref_path)).group(1)

# 폴더 내 모든 NBR tif 찾기 (기준 tif 제외)
pattern = os.path.join(nbr_dir, "*_NBR.tif")
for tif in glob.glob(pattern):
    if tif == ref_path:
        continue

    fname = os.path.basename(tif)
    match = date_re.search(fname)
    if not match:
        print(f"날짜를 인식할 수 없어 건너뜀: {fname}")
        continue
    tgt_date = match.group(1)

    # 대상 영상 열기
    with rasterio.open(tif) as tgt_src:
        # 해상도·지리정보가 기준과 다른 경우 재표본화
        if tgt_src.shape != ref_nbr.shape or tgt_src.transform != ref_meta["transform"]:
            # 최근접(nearest) 재표본화 - NBR 자료이므로 값 보존을 위해
            data = tgt_src.read(
                1,
                out_shape=ref_nbr.shape,
                resampling=Resampling.nearest,
            )
            # 새로운 transform 계산
            scale_x = tgt_src.width / ref_nbr.shape[1]
            scale_y = tgt_src.height / ref_nbr.shape[0]
            transform = tgt_src.transform * Affine.scale(scale_x, scale_y)
        else:
            data = tgt_src.read(1, masked=True)
            transform = tgt_src.transform

    # dNBR 계산
    dnbr = ref_nbr.astype(np.float32) - data.astype(np.float32)

    # 메타데이터 갱신
    out_meta = ref_meta.copy()
    out_meta.update({
        "dtype": "float32",
        "transform": transform,
        "nodata": None
    })

    out_name = f"dNBR_{ref_date}_{tgt_date}.tif"
    out_path = os.path.join(nbr_dir, out_name)

    with rasterio.open(out_path, "w", **out_meta) as dst:
        dst.write(dnbr, 1)

    print(f"저장 완료: {out_name}")


In [ ]:
# Sentinel-2에서 제공하는 구름 마스킹 & dNBR 임계값 필터링

import os
import numpy as np
import rasterio
from rasterio.enums import Resampling

# ---------------- 사용자 입력 ----------------
dnbr_path = r"<DATA_ROOT>\07_disaster\wildfire\hawaii\process\dNBR_20230729_20230818.tif"

# 구름 마스크 파일 (픽셀값 0: 구름 없음)
cloud_mask_1 = r"<DATA_ROOT>\07_disaster\wildfire\hawaii\sentinel2\unzipped\S2B_MSIL2A_20230729T210929_N0510_R057_T04QGJ_20240820T153637.SAFE\S2B_MSIL2A_20230729T210929_N0510_R057_T04QGJ_20240820T153637.SAFE\GRANULE\L2A_T04QGJ_A033402_20230729T210927\QI_DATA\MSK_CLDPRB_20m.jp2"
cloud_mask_2 = r"<DATA_ROOT>\07_disaster\wildfire\hawaii\sentinel2\unzipped\S2B_MSIL2A_20230818T210929_N0510_R057_T04QGJ_20240823T115019.SAFE\S2B_MSIL2A_20230818T210929_N0510_R057_T04QGJ_20240823T115019.SAFE\GRANULE\L2A_T04QGJ_A033688_20230818T210927\QI_DATA\MSK_CLDPRB_20m.jp2"

# 출력 마스크 경로
output_mask_path = dnbr_path.replace(".tif", "_mask.tif")

# ---------------- 처리 ----------------
# dNBR 불러오기
with rasterio.open(dnbr_path) as src:
    dnbr = src.read(1, masked=True)
    meta = src.meta.copy()
    height, width = dnbr.shape
    transform = src.transform
    crs = src.crs

# 구름 마스크 불러오기 (20m → 10m 해상도로 리샘플링)
def load_cloud_mask(mask_path, target_shape, target_transform):
    with rasterio.open(mask_path) as src:
        cloud = src.read(
            1,
            out_shape=target_shape,
            resampling=Resampling.nearest
        )
    return cloud

cloud1 = load_cloud_mask(cloud_mask_1, dnbr.shape, transform)
cloud2 = load_cloud_mask(cloud_mask_2, dnbr.shape, transform)

# 마스크 조건 적용
# (1) dNBR ≥ 0.1
# (2) 두 구름 마스크 모두 0 (구름 없음)
mask = np.where(
    (dnbr >= 0.1) & (cloud1 == 0) & (cloud2 == 0),
    1,  # 살림
    0   # 제거
).astype(np.uint8)

# 출력 메타데이터 갱신
meta.update({
    "dtype": "uint8",
    "count": 1,
    "nodata": 0
})

# 저장
with rasterio.open(output_mask_path, "w", **meta) as dst:
    dst.write(mask, 1)

print(f"[완료] 이진 마스크 저장됨: {output_mask_path}")


In [ ]:
# OmniCloudMask 적용

import os, time
from pathlib import Path
from functools import partial
from omnicloudmask import predict_from_load_func, load_s2

# ── 0. GPU ─────────────────────────────────
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
DEVICE = "cuda"

# ── 1. 입력 5개 ─────────────────────────────
scene_list = [
    r"<DATA_ROOT>\07_disaster\wildfire\hawaii\sentinel2\unzipped\S2A_MSIL2A_20230803T210931_N0510_R057_T04QGJ_20240823T144936.SAFE",
    r"<DATA_ROOT>\07_disaster\wildfire\hawaii\sentinel2\unzipped\S2A_MSIL2A_20230813T210931_N0510_R057_T04QGJ_20240801T122916.SAFE",
    r"<DATA_ROOT>\07_disaster\wildfire\hawaii\sentinel2\unzipped\S2B_MSIL2A_20230729T210929_N0510_R057_T04QGJ_20240820T153637.SAFE",
    r"<DATA_ROOT>\07_disaster\wildfire\hawaii\sentinel2\unzipped\S2B_MSIL2A_20230808T210929_N0510_R057_T04QGJ_20240822T143605.SAFE",
    r"<DATA_ROOT>\07_disaster\wildfire\hawaii\sentinel2\unzipped\S2B_MSIL2A_20230818T210929_N0510_R057_T04QGJ_20240823T115019.SAFE",
]

# ── 2. 출력 폴더 (sentinel2 내부) ─────────────
base_dir = Path(r"<DATA_ROOT>\07_disaster\wildfire\hawaii\sentinel2") / "omni_cloudmask"
MASK_DIR = base_dir / "mask"
PROB_DIR = base_dir / "prob"
MASK_DIR.mkdir(parents=True, exist_ok=True)
PROB_DIR.mkdir(parents=True, exist_ok=True)

# ── 3. 추론 ─────────────────────────────────
load_s2_20m = partial(load_s2, resolution=20.0)
t0 = time.time()

print(f"[MASK]  → {MASK_DIR}")
predict_from_load_func(
    scene_paths=scene_list,
    load_func=load_s2_20m,
    export_confidence=False,      # 0/1
    output_dir=MASK_DIR,
    batch_size=1,
    inference_device=DEVICE,
    inference_dtype="bf16",
)

print(f"[PROB]  → {PROB_DIR}")
predict_from_load_func(
    scene_paths=scene_list,
    load_func=load_s2_20m,
    export_confidence=True,       # 0–100
    output_dir=PROB_DIR,
    batch_size=1,
    inference_device=DEVICE,
    inference_dtype="bf16",
)

print("✅ 완료 – 소요:", time.strftime("%H:%M:%S", time.gmtime(time.time() - t0)))


In [ ]:
# omni_cloudmask 디렉터리 아래의 *.tif.msk 파일을 모두 삭제

import os, glob
from pathlib import Path

# ── 지울 대상 폴더 ───────────────────────────
base_dir = Path(r"<DATA_ROOT>\07_disaster\wildfire\hawaii\sentinel2\omni_cloudmask")

deleted = 0
for msk_path in glob.glob(str(base_dir / "**" / "*.tif.msk"), recursive=True):
    try:
        os.remove(msk_path)
        deleted += 1
    except Exception as e:
        print(f"삭제 실패: {msk_path} → {e}")

print(f"🗑️  .tif.msk 파일 {deleted}개 삭제 완료")


In [ ]:
# OmniCloudMask 결과로 구름 마스킹 & dNBR 임계값 필터링
# use_cloudx = True  --> Cloud_X.shp(연기) 영역을 Clear(0) 로 강제 -> 20230808 날짜에만 적용
# use_land   = True  --> land.shp 영역 *내부* 픽셀만 사용
# 불 : dNBR 임계값(0.2) / 피해지역 : dNBR 임계값(0.1)


import re, glob
import numpy as np
import rasterio
from rasterio.enums import Resampling
from rasterio import features
import geopandas as gpd
from pathlib import Path

# ── 1. 입력 dNBR ─────────────────────────────
dnbr_path = r"<DATA_ROOT>\07_disaster\wildfire\hawaii\process\dNBR_20230803_20230818.tif"

# OmniCloudMask 클래스 맵 폴더
omni_mask_dir = Path(r"<DATA_ROOT>\07_disaster\wildfire\hawaii\sentinel2\omni_cloudmask\mask")

# (선택) 예외·클리핑 옵션
use_cloudx = False   # True → Cloud_X 영역 Clear 로 강제   / False
cloudx_shp = r"<DATA_ROOT>\07_disaster\wildfire\hawaii\shp\Cloud_X.shp"

use_land   = True   # True → land.shp 내부만 결과 포함   / False
land_shp   = r"<DATA_ROOT>\07_disaster\wildfire\hawaii\shp\land.shp"

# ── 2. dNBR 파일명에서 두 날짜 추출 ───────────
m = re.search(r"dNBR_(\d{8})_(\d{8})\.tif$", Path(dnbr_path).name)
date1, date2 = m.group(1), m.group(2)

# ── 3. 해당 날짜 클래스 맵 찾기 ────────────────
def find_class(date):
    pattern = omni_mask_dir / f"*{date}*_OCM_v1_4_1.tif"
    hits = glob.glob(str(pattern))
    if not hits:
        raise FileNotFoundError(f"{date} 클래스 맵 없음 ▶ {pattern}")
    return hits[0]

mask1_path = find_class(date1)
mask2_path = find_class(date2)

# ── 4. dNBR 읽기 ─────────────────────────────
with rasterio.open(dnbr_path) as dsrc:
    dnbr        = dsrc.read(1, masked=True)
    meta        = dsrc.meta.copy()
    tgt_shape   = dnbr.shape
    transform   = dsrc.transform
    crs         = dsrc.crs

# ── 5. 클래스 맵 읽기 (20 m → dNBR 크기) ──────
def load_class(path):
    with rasterio.open(path) as src:
        return src.read(1, out_shape=tgt_shape, resampling=Resampling.nearest)

class1 = load_class(mask1_path)
class2 = load_class(mask2_path)

# ── 6. Cloud_X 예외 적용 (Clear 로 강제) ───────
if use_cloudx:
    gdf = gpd.read_file(cloudx_shp).to_crs(crs)
    exc = features.rasterize(
        [(geom, 1) for geom in gdf.geometry],
        out_shape=tgt_shape, transform=transform, fill=0, dtype=np.uint8
    )
    class1[exc == 1] = 0
    class2[exc == 1] = 0

# ── 7. land.shp 클리핑 (내부만 사용) ──────────
if use_land:
    land_gdf = gpd.read_file(land_shp).to_crs(crs)
    land_mask = features.rasterize(
        [(geom, 1) for geom in land_gdf.geometry],
        out_shape=tgt_shape, transform=transform, fill=0, dtype=np.uint8
    )
else:
    land_mask = np.ones(tgt_shape, dtype=np.uint8)

# ── 8. dNBR ≥ 임계값 & Clear & land ─────────────
binary = np.where(
    (dnbr >= 0.1) & (class1 == 0) & (class2 == 0) & (land_mask == 1),
    1, 0
).astype(np.uint8)

# ── 9. 출력 파일명 구성 ───────────────────────
suffix_parts = ["_Omni_mask"]
if use_cloudx:
    suffix_parts.append("_CloudX")
if use_land:
    suffix_parts.append("_land")
out_path = Path(dnbr_path).with_name(Path(dnbr_path).stem + "".join(suffix_parts) + ".tif")

# ── 10. 저장 ─────────────────────────────────
meta.update({"dtype": "uint8", "count": 1, "nodata": 0})
with rasterio.open(out_path, "w", **meta) as dst:
    dst.write(binary, 1)

print("✅ 저장 완료:", out_path)


In [ ]:
# OmniCloudMask 결과로 구름 마스킹 & dNBR 임계값 필터링
# use_cloudx = True  --> Cloud_X.shp(연기) 영역을 Clear(0) 로 강제 -> 20230808 날짜에만 적용
# use_land   = True  --> land.shp 영역 *내부* 픽셀만 사용
# 불 : dNBR 임계값(0.2) / 피해지역 : dNBR 임계값(0.1)
# 불 : NBR 임계값(-0.4 이하) / 피해지역 : NBR 임계값(-0.05 이하)

import re, glob
import numpy as np
import rasterio
from rasterio.enums import Resampling
from rasterio import features
import geopandas as gpd
from pathlib import Path

# ───────────── 사용자 설정 (꼼꼼히 확인할 것!!!!)─────────────
DNBRT_THRESH = 0.1       # dNBR ≥ ?
use_cloudx   = False     # 연기 지역 마스킹 (20230808 날짜에만 적용)    True    False
use_land     = True      # land.shp 내부만 사용 (항상)
use_nbr      = True      # NBR 임계값 필터 적용 여부
NBR_THRESH   = -0.05     # NBR < ? 중 한 날짜라도 → 살림

dnbr_path = r"<DATA_ROOT>\07_disaster\wildfire\hawaii\process\dNBR_20230729_20230818.tif"

omni_mask_dir = Path(r"<DATA_ROOT>\07_disaster\wildfire\hawaii\sentinel2\omni_cloudmask\mask")
cloudx_shp = r"<DATA_ROOT>\07_disaster\wildfire\hawaii\shp\Cloud_X.shp"
land_shp   = r"<DATA_ROOT>\07_disaster\wildfire\hawaii\shp\land.shp"
nbr_dir    = Path(r"<DATA_ROOT>\07_disaster\wildfire\hawaii\process")   # NBR tif 위치


# ── 1. 날짜 추출 ──────────────────────────────
m = re.search(r"dNBR_(\d{8})_(\d{8})\.tif$", Path(dnbr_path).name)
date1, date2 = m.group(1), m.group(2)

# ── 2. 클래스 맵 경로 ─────────────────────────
def find_class(d):
    hits = glob.glob(str(omni_mask_dir / f"*{d}*_OCM_v1_4_1.tif"))
    if not hits:
        raise FileNotFoundError(f"{d} 클래스 맵 없음")
    return hits[0]
mask1_path, mask2_path = map(find_class, [date1, date2])

# ── 3. dNBR 읽기 ─────────────────────────────
with rasterio.open(dnbr_path) as dsrc:
    dnbr = dsrc.read(1, masked=True)
    meta, tgt_shape, transform, crs = dsrc.meta.copy(), dsrc.shape, dsrc.transform, dsrc.crs

# ── 4. 클래스 맵 읽기 ─────────────────────────
def load_class(p):
    with rasterio.open(p) as s:
        return s.read(1, out_shape=tgt_shape, resampling=Resampling.nearest)
class1, class2 = map(load_class, [mask1_path, mask2_path])

# ── 5. Cloud_X 예외 ──────────────────────────
if use_cloudx:
    exc = features.rasterize(
        [(g, 1) for g in gpd.read_file(cloudx_shp).to_crs(crs).geometry],
        out_shape=tgt_shape, transform=transform, fill=0, dtype=np.uint8
    )
    class1[exc == 1] = class2[exc == 1] = 0  # Clear 강제

# ── 6. land.shp 클리핑 ───────────────────────
land_mask = np.ones(tgt_shape, np.uint8)
if use_land:
    land_mask = features.rasterize(
        [(g, 1) for g in gpd.read_file(land_shp).to_crs(crs).geometry],
        out_shape=tgt_shape, transform=transform, fill=0, dtype=np.uint8
    )

# ── 7. (선택) NBR 파일 읽어 임계값 적용 ──────
if use_nbr:
    def load_nbr(d):
        p = nbr_dir / f"{d}_T04QGJ_NBR.tif"
        with rasterio.open(p) as s:
            return s.read(1, out_shape=tgt_shape, resampling=Resampling.nearest)
    nbr1, nbr2 = map(load_nbr, [date1, date2])
    nbr_mask = (nbr1 < NBR_THRESH) | (nbr2 < NBR_THRESH)
else:
    nbr_mask = np.ones(tgt_shape, bool)

# ── 8. 최종 조건 & 이진화 ────────────────────
binary = np.where(
    (dnbr >= DNBRT_THRESH) & (class1 == 0) & (class2 == 0) & land_mask & nbr_mask,
    1, 0
).astype(np.uint8)

# ── 9. 출력 파일명 ───────────────────────────
suffix = ["_Omni_mask"]
if use_cloudx: suffix.append("_CloudX")
if use_land:   suffix.append("_land")
if use_nbr:    suffix.append(f"_NBR{NBR_THRESH}")
out_path = Path(dnbr_path).with_name(Path(dnbr_path).stem + "".join(suffix) + ".tif")

# ── 10. 저장 ─────────────────────────────────
meta.update({"dtype": "uint8", "count": 1, "nodata": 0})
with rasterio.open(out_path, "w", **meta) as dst:
    dst.write(binary, 1)

print("✅ 저장 완료:", out_path)


In [ ]:
# dNBR 임계값 ≤ 0.27  & Lahaina.shp 내부 픽셀만  dNDBI 값 저장


import rasterio
import numpy as np
import geopandas as gpd
from rasterio.enums import Resampling
from rasterio.warp import reproject
from rasterio import features
from pathlib import Path

# ── 입력 경로 ─────────────────────────────
dnbr_tif   = r"<DATA_ROOT>\07_disaster\wildfire\hawaii\process\dNBR_20230729_20230803.tif"
dndbi_tif  = r"<DATA_ROOT>\07_disaster\wildfire\hawaii\process\dNDBI_20230729_20230803.tif"
lahaina_shp = r"<DATA_ROOT>\07_disaster\wildfire\hawaii\shp\land.shp"

# ── 출력 경로 ────────────────────────────
out_tif = Path(dnbr_tif).with_name("Risk_20230729_20230803.tif")

# 1. dNBR 읽기 (기준 해상도·변환 확보)
with rasterio.open(dnbr_tif) as src_nbr:
    dnbr      = src_nbr.read(1, masked=True).astype(np.float32)
    profile   = src_nbr.profile
    tr, crs   = src_nbr.transform, src_nbr.crs
    shape     = src_nbr.shape

# 2. dNDBI 읽기 → dNBR 해상도로 맞춤
with rasterio.open(dndbi_tif) as src_ndbi:
    if (src_ndbi.transform != tr) or (src_ndbi.shape != shape):
        dndbi = np.empty(shape, dtype=np.float32)
        reproject(
            source=src_ndbi.read(1),
            destination=dndbi,
            src_transform=src_ndbi.transform, src_crs=src_ndbi.crs,
            dst_transform=tr,              dst_crs=crs,
            resampling=Resampling.nearest
        )
    else:
        dndbi = src_ndbi.read(1, masked=True).astype(np.float32)

# 3. Lahaina.shp → 래스터 마스크(1=내부, 0=외부)
gdf = gpd.read_file(lahaina_shp).to_crs(crs)
lahaina_mask = features.rasterize(
    [(geom, 1) for geom in gdf.geometry],
    out_shape=shape,
    transform=tr,
    fill=0,
    dtype=np.uint8
)

# 4. 조건 적용: (dNBR ≤ 0.27) & (Lahaina 내부)
mask_cond = (dnbr <= 0.27) & (lahaina_mask == 1)
result    = np.where(mask_cond, dndbi, np.nan).astype(np.float32)

# 5. 저장
profile.update(driver="GTiff", dtype="float32", count=1, nodata=np.nan)
with rasterio.open(out_tif, "w", **profile) as dst:
    dst.write(result, 1)

print("✅ 저장 완료 →", out_tif)


In [ ]:
# dNBR ≤ 0.27  AND  Lahaina.shp 내부  AND  두 날짜 모두 Cloud‑free
# → dNDBI 값만 남겨 저장


import rasterio, numpy as np, geopandas as gpd, glob, re
from rasterio.enums import Resampling
from rasterio.warp import reproject
from rasterio import features
from pathlib import Path

# ─────────── 사용자 입력 ─────────────────────────────────────
dnbr_tif   = r"<DATA_ROOT>\07_disaster\wildfire\hawaii\process\dNBR_20230729_20230813.tif"
dndbi_tif  = r"<DATA_ROOT>\07_disaster\wildfire\hawaii\process\dNDBI_20230729_20230813.tif"
out_tif    = Path(dnbr_tif).with_name("Risk_20230729_20230813_La.tif")

shp_path   = r"<DATA_ROOT>\07_disaster\wildfire\hawaii\shp\Lahaina.shp"
omni_dir   = Path(r"<DATA_ROOT>\07_disaster\wildfire\hawaii\sentinel2\omni_cloudmask\mask")

# ─────────────────────────────────────────────────────────────

# 1) 날짜 추출 (YYYYMMDD)
m = re.search(r"dNBR_(\d{8})_(\d{8})", Path(dnbr_tif).name)
date1, date2 = m.group(1), m.group(2)

# 2) 함수: 클래스맵 → dNBR 그리드로 읽기 (Clear=0)
def load_cloud(date, ref_shape, ref_tr, ref_crs):
    cm_path = glob.glob(str(omni_dir / f"*{date}*_OCM_v1_4_1.tif"))[0]
    with rasterio.open(cm_path) as src:
        if (src.transform != ref_tr) or (src.shape != ref_shape):
            dst = np.empty(ref_shape, dtype=np.uint8)
            reproject(
                source=src.read(1),
                destination=dst,
                src_transform=src.transform, src_crs=src.crs,
                dst_transform=ref_tr,        dst_crs=ref_crs,
                resampling=Resampling.nearest
            )
        else:
            dst = src.read(1)
    return dst

# 3) dNBR 읽기 (기준 격자)
with rasterio.open(dnbr_tif) as s:
    dnbr   = s.read(1, masked=True).astype(np.float32)
    prof   = s.profile
    tr, crs, shape = s.transform, s.crs, s.shape

# 4) dNDBI 읽기 → dNBR 격자로 맞춤
with rasterio.open(dndbi_tif) as s2:
    if (s2.transform != tr) or (s2.shape != shape):
        dndbi = np.empty(shape, dtype=np.float32)
        reproject(s2.read(1), dndbi,
                  src_transform=s2.transform, src_crs=s2.crs,
                  dst_transform=tr,          dst_crs=crs,
                  resampling=Resampling.nearest)
    else:
        dndbi = s2.read(1, masked=True).astype(np.float32)

# 5) Lahaina / land 영역 마스크
mask_land = features.rasterize(
    [(g, 1) for g in gpd.read_file(shp_path).to_crs(crs).geometry],
    out_shape=shape, transform=tr, fill=0, dtype=np.uint8)

# 6) OmniCloudMask 클래스맵 → 두 날짜 모두 Clear(=0)
cloud1 = load_cloud(date1, shape, tr, crs)  # 클래스 0 = Clear
cloud2 = load_cloud(date2, shape, tr, crs)
mask_cloud = (cloud1 == 0) & (cloud2 == 0)

# 7) 최종 조건: dNBR ≤ 0.27 & land & cloud‑free
mask_cond = (dnbr <= 0.27) & (mask_land == 1) & mask_cloud
result    = np.where(mask_cond, dndbi, np.nan).astype(np.float32)

# 8) 저장
prof.update(driver="GTiff", dtype="float32", count=1, nodata=np.nan)
with rasterio.open(out_tif, "w", **prof) as dst:
    dst.write(result, 1)

print("✅ 저장 완료 →", out_tif)
print("   * 조건: dNBR ≤ 0.27, land.shp 내부, 두 날짜 모두 Clear")


In [ ]:
# NDVI 임계값 < 0.3  & Lahaina.shp 내부 픽셀만  dNBR 값 저장

import rasterio, numpy as np, geopandas as gpd, glob, re
from rasterio.enums import Resampling
from rasterio.warp import reproject
from rasterio import features
from pathlib import Path

# ─────────── 사용자 입력 ─────────────────────────────────────
ndvi_path   = r"<DATA_ROOT>\07_disaster\wildfire\hawaii\process\20230729_T04QGJ_NDVI.tif"   # 필터 기준
dnbr_path   = r"<DATA_ROOT>\07_disaster\wildfire\hawaii\process\dNBR_20230729_20230813.tif"  # 실제 출력값

# NDVI 임계값 포함된 출력 파일명 생성
NDVI_THRESHOLD = 0.3    # NDVI < NDVI_THRESHOLD
ndvi_label = f"NDVI{NDVI_THRESHOLD:.2f}".rstrip("0").rstrip(".")  # 예: 0.30 → NDVI0.3
out_path   = Path(dnbr_path).with_name(f"Risk_dNBR_20230729_20230813_La_{ndvi_label}.tif")

shp_path   = r"<DATA_ROOT>\07_disaster\wildfire\hawaii\shp\Lahaina.shp"
omni_dir   = Path(r"<DATA_ROOT>\07_disaster\wildfire\hawaii\sentinel2\omni_cloudmask\mask")
# ─────────────────────────────────────────────────────────────

# 1) 날짜 추출 (YYYYMMDD)
date1 = re.search(r"(\d{8})", Path(ndvi_path).name).group(1)
date2 = re.search(r"(\d{8})_(\d{8})", Path(dnbr_path).name).group(2)

# 2) 함수: 구름 마스크 → dNBR 격자에 맞춰 읽기
def load_cloud(date, ref_shape, ref_tr, ref_crs):
    cm_path = glob.glob(str(omni_dir / f"*{date}*_OCM_v1_4_1.tif"))[0]
    with rasterio.open(cm_path) as src:
        if (src.transform != ref_tr) or (src.shape != ref_shape):
            dst = np.empty(ref_shape, dtype=np.uint8)
            reproject(
                source=src.read(1),
                destination=dst,
                src_transform=src.transform, src_crs=src.crs,
                dst_transform=ref_tr,        dst_crs=ref_crs,
                resampling=Resampling.nearest
            )
        else:
            dst = src.read(1)
    return dst

# 3) dNBR 읽기 (기준 격자)
with rasterio.open(dnbr_path) as src_dnbr:
    dnbr = src_dnbr.read(1).astype(np.float32)
    profile = src_dnbr.profile
    tr, crs, shape = src_dnbr.transform, src_dnbr.crs, src_dnbr.shape

# 4) NDVI → dNBR 격자로 재배치
with rasterio.open(ndvi_path) as src_ndvi:
    if (src_ndvi.transform != tr) or (src_ndvi.shape != shape):
        ndvi = np.empty(shape, dtype=np.float32)
        reproject(
            source=src_ndvi.read(1),
            destination=ndvi,
            src_transform=src_ndvi.transform, src_crs=src_ndvi.crs,
            dst_transform=tr,               dst_crs=crs,
            resampling=Resampling.nearest
        )
    else:
        ndvi = src_ndvi.read(1).astype(np.float32)

# 5) Lahaina.shp → 래스터 마스크 (1 = 내부)
gdf = gpd.read_file(shp_path).to_crs(crs)
mask_land = features.rasterize(
    [(geom, 1) for geom in gdf.geometry],
    out_shape=shape,
    transform=tr,
    fill=0,
    dtype=np.uint8
)

# 6) 두 날짜 모두 Clear(=0) 영역 추출
cloud1 = load_cloud(date1, shape, tr, crs)
cloud2 = load_cloud(date2, shape, tr, crs)
mask_cloud = (cloud1 == 0) & (cloud2 == 0)

# 7) 조건: NDVI < NDVI_THRESHOLD & Lahaina & cloud-free
mask = (ndvi < NDVI_THRESHOLD) & (mask_land == 1) & mask_cloud
result = np.where(mask, dnbr, np.nan).astype(np.float32)  # dNBR 값은 원본 그대로 유지됨

# 8) 저장
profile.update(driver="GTiff", dtype="float32", count=1, nodata=np.nan)
with rasterio.open(out_path, "w", **profile) as dst:
    dst.write(result, 1)

print("✅ 저장 완료 →", out_path)
print(f"   * 조건: NDVI < {NDVI_THRESHOLD}, Lahaina 영역, 두 날짜 모두 Clear")


In [ ]:
# 어떤 파일이든 구름, shp 영역 내만 마스킹
# use_cloudx = True  --> Cloud_X.shp(연기) 영역을 Clear(0) 로 강제 -> 20230808 날짜에만 적용

import rasterio, numpy as np, geopandas as gpd, glob, re
from rasterio.enums import Resampling
from rasterio.warp import reproject
from rasterio import features
from pathlib import Path

# ─────────── 사용자 입력 ─────────────────────────────────────
in_tif      = r"<DATA_ROOT>\07_disaster\wildfire\hawaii\process\20230729_T04QGJ_BAIS2.tif"
shp_path    = r"<DATA_ROOT>\07_disaster\wildfire\hawaii\shp\land.shp"
omni_dir    = Path(r"<DATA_ROOT>\07_disaster\wildfire\hawaii\sentinel2\omni_cloudmask\mask")

use_cloudx  = False      # True  False
cloudx_shp  = r"<DATA_ROOT>\07_disaster\wildfire\hawaii\shp\Cloud_X.shp"
# ─────────────────────────────────────────────────────────────

# 1) 날짜 추출 (YYYYMMDD)
date = re.search(r"(\d{8})", Path(in_tif).name).group(1)

# 2) 입력 영상 읽기
with rasterio.open(in_tif) as src:
    data = src.read(1).astype(np.float32)
    profile = src.profile
    tr, crs, shape = src.transform, src.crs, src.shape

# 3) shp_path 영역 마스크
mask_land = features.rasterize(
    [(g, 1) for g in gpd.read_file(shp_path).to_crs(crs).geometry],
    out_shape=shape,
    transform=tr,
    fill=0,
    dtype=np.uint8
)

# 4) 구름 마스크 → 입력 영상 격자에 맞춤
def load_cloud(date, ref_shape, ref_tr, ref_crs):
    cm_path = glob.glob(str(omni_dir / f"*{date}*_OCM_v1_4_1.tif"))[0]
    with rasterio.open(cm_path) as src:
        if (src.transform != ref_tr) or (src.shape != ref_shape):
            dst = np.empty(ref_shape, dtype=np.uint8)
            reproject(
                source=src.read(1),
                destination=dst,
                src_transform=src.transform, src_crs=src.crs,
                dst_transform=ref_tr,        dst_crs=ref_crs,
                resampling=Resampling.nearest
            )
        else:
            dst = src.read(1)
    return dst

cloud_mask = load_cloud(date, shape, tr, crs)  # Clear = 0

# 5) Cloud_X 영역 → Clear로 강제 (선택적)
if use_cloudx:
    gdf_cloudx = gpd.read_file(cloudx_shp).to_crs(crs)
    mask_cloudx = features.rasterize(
        [(g, 1) for g in gdf_cloudx.geometry],
        out_shape=shape,
        transform=tr,
        fill=0,
        dtype=np.uint8
    )
    cloud_mask[mask_cloudx == 1] = 0  # Clear로 간주

# 6) 마스킹 조건 적용
mask = (cloud_mask == 0) & (mask_land == 1)
result = np.where(mask, data, np.nan).astype(np.float32)

# 7) 출력 경로 설정 (CloudX 여부 반영)
suffix = "_CloudX" if use_cloudx else ""
out_tif = str(Path(in_tif).with_name(Path(in_tif).stem + "_mask" + suffix + ".tif"))

# 8) 저장
profile.update(driver="GTiff", dtype="float32", count=1, nodata=np.nan)
with rasterio.open(out_tif, "w", **profile) as dst:
    dst.write(result, 1)

print("✅ 저장 완료 →", out_tif)
print("   * 조건: shp 내부 & Clear(0)" + (" (Cloud_X 영역 포함)" if use_cloudx else ""))


In [ ]:
# 어떤 파일이든 어떤 shp 영역 내에 있는 값만 남기기

import rasterio
import numpy as np
import geopandas as gpd
from rasterio import features
from pathlib import Path
import os
from glob import glob

# ───────────── 사용자 입력 ─────────────
# 아래 둘 중 하나만 설정
tif_file = None  # 단일 파일 (예: r"D:\...\a.tif")
tif_folder = r"<DATA_ROOT>\07_disaster\wildfire\Greece\Evia\process"  # 전체 폴더

# SHP 경로 (클리핑할 영역)
shp_path = r"<DATA_ROOT>\07_disaster\wildfire\Greece\shp\grc_adm_2025_ab_shp\grc_admbnda_adm0_AB.shp"

# ───────────── 함수 정의 ─────────────
def mask_tif_with_shp(tif_path, shp_path):
    tif_path = Path(tif_path)
    out_path = tif_path.with_name(tif_path.stem + "_masked.tif")

    # 1. TIF 읽기
    with rasterio.open(tif_path) as src:
        data = src.read(1).astype(np.float32)
        profile = src.profile
        transform = src.transform
        crs = src.crs
        shape = src.shape
        nodata_val = src.nodata if src.nodata is not None else -9999

    # 2. Shapefile 읽고 좌표계 맞춤
    gdf = gpd.read_file(shp_path).to_crs(crs)

    # 3. 래스터 마스크 생성 (1 = 내부)
    mask = features.rasterize(
        [(geom, 1) for geom in gdf.geometry],
        out_shape=shape,
        transform=transform,
        fill=0,
        dtype=np.uint8
    )

    # 4. 마스크 적용
    masked_data = np.where(mask == 1, data, nodata_val).astype(np.float32)

    # 5. 저장
    profile.update(driver="GTiff", dtype="float32", count=1, nodata=nodata_val)
    with rasterio.open(out_path, "w", **profile) as dst:
        dst.write(masked_data, 1)

    print(f"✅ 마스킹 완료: {out_path.name}")

# ───────────── 실행 로직 ─────────────
if tif_file:
    mask_tif_with_shp(tif_file, shp_path)

elif tif_folder:
    tif_list = sorted(glob(os.path.join(tif_folder, "*.tif")))
    if not tif_list:
        print("❌ TIF 파일이 폴더에 없습니다.")
    else:
        for tif in tif_list:
            mask_tif_with_shp(tif, shp_path)

else:
    print("❌ tif_file 또는 tif_folder 중 하나를 설정하세요.")






In [ ]:
# NBR이 0보다 작고 dNBR이 0.1 이상인 부분만 남기는 마스크 생성

import rasterio
import numpy as np
from pathlib import Path

# ───────────── 사용자 입력 ─────────────
dnbr_path = r"<DATA_ROOT>\07_disaster\wildfire\Greece\Kythira\process\dNBR_20250725_20250728_masked.tif"
nbr_after_path = r"<DATA_ROOT>\07_disaster\wildfire\Greece\Kythira\process\20250728_T34SFF_NBR_masked.tif"

USE_NBR_FILTER = True         # True → NBR < 0 조건도 함께 적용
DNBR_THRESHOLD = 0.1

# 출력 경로 설정
output_tif = Path(dnbr_path).with_name(Path(dnbr_path).stem + "_binary.tif")

# ───────────── dNBR 읽기 ─────────────
with rasterio.open(dnbr_path) as src_dnbr:
    dnbr = src_dnbr.read(1).astype(np.float32)
    profile = src_dnbr.profile.copy()

# ───────────── 조건 1: dNBR ≥ 0.1 ─────────────
mask = (dnbr >= DNBR_THRESHOLD)

# ───────────── 조건 2: NBR < 0 (불난 이후) ─────────────
if USE_NBR_FILTER:
    with rasterio.open(nbr_after_path) as src_nbr:
        nbr_after = src_nbr.read(1).astype(np.float32)
    mask = mask & (nbr_after < 0)

# ───────────── 이진 마스크 생성 및 저장 ─────────────
binary_mask = np.where(mask, 1, 0).astype(np.uint8)

profile.update({
    "dtype": "uint8",
    "count": 1,
    "nodata": 0
})

with rasterio.open(output_tif, "w", **profile) as dst:
    dst.write(binary_mask, 1)

print(f"✅ 이진 마스크 저장 완료:\n{output_tif}")



In [ ]:
# 어떤 파일이든 어떤 마스크 위에 있는 값만 오버레이

import rasterio
import numpy as np
from pathlib import Path

# ─────────── 사용자 입력 ─────────────────────────────────────
dnbr_path  = r"<DATA_ROOT>\07_disaster\wildfire\hawaii\process\dNBR_20230729_20230818.tif"
mask_path  = r"<DATA_ROOT>\07_disaster\wildfire\hawaii\process\dNBR_20230729_20230818_Omni_mask_land_NBR-0.05.tif"
out_path   = str(Path(dnbr_path).with_name(Path(dnbr_path).stem + "_overlay.tif"))
# ─────────────────────────────────────────────────────────────

# 1) dNBR 읽기
with rasterio.open(dnbr_path) as src_dnbr:
    dnbr = src_dnbr.read(1).astype(np.float32)
    profile = src_dnbr.profile

# 2) 마스크 읽기
with rasterio.open(mask_path) as src_mask:
    mask = src_mask.read(1).astype(np.uint8)

# 3) 마스킹 처리: mask==1인 부분만 dNBR 유지, 나머지는 NaN
masked_dnbr = np.where(mask == 1, dnbr, np.nan).astype(np.float32)

# 4) 저장
profile.update(driver="GTiff", dtype="float32", count=1, nodata=np.nan)
with rasterio.open(out_path, "w", **profile) as dst:
    dst.write(masked_dnbr, 1)

print("✅ 저장 완료 →", out_path)
print("   * 조건: 마스크 값이 1인 부분만 dNBR 유지")
